<a href="https://colab.research.google.com/github/rushi-k-op/EV_battery_analytics_ml/blob/main/EV_battery_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import scipy.io as sio  # This is crucial for reading the .mat files
import matplotlib.pyplot as plot

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!unzip "/content/drive/MyDrive/battery_alt_dataset.zip" -d "/content/battery_data/"

Archive:  /content/drive/MyDrive/battery_alt_dataset.zip
   creating: /content/battery_data/battery_alt_dataset/
  inflating: /content/battery_data/__MACOSX/._battery_alt_dataset  
   creating: /content/battery_data/battery_alt_dataset/regular_alt_batteries/
  inflating: /content/battery_data/__MACOSX/battery_alt_dataset/._regular_alt_batteries  
   creating: /content/battery_data/battery_alt_dataset/recommissioned_batteries/
  inflating: /content/battery_data/__MACOSX/battery_alt_dataset/._recommissioned_batteries  
  inflating: /content/battery_data/battery_alt_dataset/README.txt  
  inflating: /content/battery_data/__MACOSX/battery_alt_dataset/._README.txt  
  inflating: /content/battery_data/battery_alt_dataset/LICENSE.txt  
  inflating: /content/battery_data/__MACOSX/battery_alt_dataset/._LICENSE.txt  
   creating: /content/battery_data/battery_alt_dataset/second_life_batteries/
  inflating: /content/battery_data/__MACOSX/battery_alt_dataset/._second_life_batteries  
  inflating: 

In [5]:
# Let's pick the first battery from the regular dataset to inspect
file_path = "/content/battery_data/battery_alt_dataset/regular_alt_batteries/battery00.csv"

# Read the CSV file into a pandas DataFrame (think of this as a Python spreadsheet)
df = pd.read_csv(file_path)

# 1. Print the column names and data types to see what we are working with
print("--- Dataset Information ---")
df.info()

# 2. Display the first 5 rows of the data
print("\n--- First 5 Rows of Data ---")
display(df.head())

--- Dataset Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1101244 entries, 0 to 1101243
Data columns (total 10 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   start_time            1101244 non-null  object 
 1   time                  1101244 non-null  float64
 2   mode                  1101244 non-null  float64
 3   voltage_charger       1101244 non-null  float64
 4   temperature_battery   1101244 non-null  float64
 5   voltage_load          106006 non-null   float64
 6   current_load          106006 non-null   float64
 7   temperature_mosfet    106006 non-null   float64
 8   temperature_resistor  106006 non-null   float64
 9   mission_type          106006 non-null   float64
dtypes: float64(9), object(1)
memory usage: 84.0+ MB

--- First 5 Rows of Data ---


,start_time,time,mode,voltage_charger,temperature_battery,voltage_load,current_load,temperature_mosfet,temperature_resistor,mission_type
0,2022-07-19 11:10:00,0.000,0.0,0.000,0.000,NaN,NaN,NaN,NaN,NaN
1,2022-07-19 11:10:00,1.894,0.0,8.341,23.059,NaN,NaN,NaN,NaN,NaN
2,2022-07-19 11:10:00,2.814,0.0,8.340,23.059,NaN,NaN,NaN,NaN,NaN
3,2022-07-19 11:10:00,3.734,0.0,8.341,23.063,NaN,NaN,NaN,NaN,NaN
4,2022-07-19 11:10:00,4.654,0.0,8.341,23.063,NaN,NaN,NaN,NaN,NaN


In [6]:
# 1. Let's see all the different "phases" the battery goes through
print("Unique operating modes in the dataset:")
print(df['mode'].unique())

# 2. Let's filter out all the rows where the load sensors were turned off (the NaNs)
df_discharging = df.dropna(subset=['current_load'])

# 3. Let's see what the data looks like when the battery is actually being used
print(f"\nRemaining rows after dropping NaNs: {len(df_discharging)}")
display(df_discharging.head())

Unique operating modes in the dataset:
[ 0. -1.  1.]

Remaining rows after dropping NaNs: 106006


,start_time,time,mode,voltage_charger,temperature_battery,voltage_load,current_load,temperature_mosfet,temperature_resistor,mission_type
324,2022-07-19 11:10:00,300.523,-1.0,8.340,22.963,-0.026,0.152,22.64,23.20,0.0
325,2022-07-19 11:10:00,301.463,-1.0,8.041,23.155,8.329,2.523,22.65,23.21,0.0
326,2022-07-19 11:10:00,302.404,-1.0,8.030,23.170,8.320,2.521,22.66,23.20,0.0
327,2022-07-19 11:10:00,303.345,-1.0,8.023,23.186,8.312,2.523,22.68,23.21,0.0
328,2022-07-19 11:10:00,304.288,-1.0,8.016,23.201,8.305,2.521,22.71,23.21,0.0


In [7]:
# 1. Sort the data to ensure time is in order
df_discharging = df_discharging.sort_values(by=['start_time', 'time'])

# 2. Calculate the time difference (dt) between readings in HOURS
# (Time is in seconds, so we divide by 3600)
df_discharging['dt_hours'] = df_discharging.groupby('start_time')['time'].diff().fillna(0) / 3600.0

# 3. Calculate the Ampere-hour (Ah) slice for each row
df_discharging['capacity_slice'] = df_discharging['current_load'].abs() * df_discharging['dt_hours']

# 4. Group everything by the "start_time" (which represents one cycle)
cycle_data = df_discharging.groupby('start_time').agg(
    total_capacity_Ah=('capacity_slice', 'sum'),
        max_temp=('temperature_battery', 'max'),
            avg_voltage=('voltage_load', 'mean')
            ).reset_index()

# 5. Sort by time so we can see the battery aging
cycle_data = cycle_data.sort_values(by='start_time').reset_index(drop=True)

print(f"Successfully extracted {len(cycle_data)} distinct discharge cycles!")
print("\n--- Cycle Summary Data ---")
display(cycle_data.head())

Successfully extracted 16 distinct discharge cycles!

--- Cycle Summary Data ---


,start_time,total_capacity_Ah,max_temp,avg_voltage
0,2022-07-19 11:10:00,2.452879,29.161,7.257751
1,2022-07-22 10:42:00,8.617332,97.022,6.456338
2,2022-07-23 12:31:00,48.109065,102.308,6.424835
3,2022-07-24 22:39:00,34.149920,97.753,6.776607
4,2022-07-26 22:45:00,45.482906,97.368,6.647994


In [8]:
from sklearn.model_selection import train_test_split

# 1. Add a 'cycle_number' column.
cycle_data['cycle_number'] = cycle_data.index + 1

# 2. Define Features (X) and Target (y)
X = cycle_data[['cycle_number', 'max_temp', 'avg_voltage']]
y = cycle_data['total_capacity_Ah']

# 3. Split the data!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total recorded cycles: {len(cycle_data)}")
print(f"Data given to the model to learn (X_train): {len(X_train)} cycles")
print(f"Data hidden for the final exam (X_test): {len(X_test)} cycles")

Total recorded cycles: 16
Data given to the model to learn (X_train): 12 cycles
Data hidden for the final exam (X_test): 4 cycles


In [9]:
from sklearn.ensemble import RandomForestRegressor

# 1. Initialize the model (We are asking for 100 'trees' in our forest)
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 2. Train the model using the 'fit' command!
print("Feeding data to the Random Forest...")
model.fit(X_train, y_train)

print("Model trained successfully!")

Feeding data to the Random Forest...
Model trained successfully!


In [10]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

# 1. Ask the model to predict the capacity for our hidden test data
predictions = model.predict(X_test)

# 2. Calculate the Mean Absolute Error
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error: {mae:.5f} Ah")
print(f"(This means our model's predictions are, on average, {mae:.5f} Ah away from the true capacity)")

# 3. Create a clean table to see the predictions side-by-side with reality
results = pd.DataFrame({
    'Cycle Number': X_test['cycle_number'],
    'Actual Capacity (Ah)': y_test,
    'Predicted Capacity (Ah)': predictions
})

# Let's calculate the exact error for each individual prediction
results['Error (Ah)'] = abs(results['Actual Capacity (Ah)'] - results['Predicted Capacity (Ah)'])

print("\n--- Final Exam Results ---")
display(results)

Mean Absolute Error: 22.17458 Ah
(This means our model's predictions are, on average, 22.17458 Ah away from the true capacity)

--- Final Exam Results ---


,Cycle Number,Actual Capacity (Ah),Predicted Capacity (Ah),Error (Ah)
0,1,2.452879,38.627849,36.174970
1,2,8.617332,43.913474,35.296143
5,6,60.090018,43.273902,16.816117
14,15,26.185682,26.596758,0.411076
